# Hidden Markov Model
*Characterized by three matrices and two concepts*
## **Concepts**:
1. Hidden states ($S$)
2. Observations ($O$)

## Matrices
1. Transition probability matrix ($A$)
2. Emission probability matrix ($B$)
3. Initial state distribution matrix ($π$)

### 1. Transition probability matrix ($A$)
- Defines the probability of transitioning from one hidden state to another.
- $A_{ij} = P(S_t = s_j \mid S_{t-1} = s_i)$
  - The probability of transitioning from state $s_i$ to state $s_j$ at time step $t$.
- Each row represents transitions starting from a specific hidden state, so each row must sum to 1:
  $$\sum_{j=1}^{N} A_{ij} = 1 \quad \forall i$$

### 2. Emission probability matrix ($B$)
- Defines the conditional probability of emitting (generating) a specific observation given a hidden state.
- $B_{ik} = P(O_t = o_k \mid S_t = s_i)$
  - The probability of observing $o_k$ when the current hidden state is $s_i$.
- Each row represents emission probabilities for a given state, so each row must sum to 1:
  $$\sum_{k=1}^{M} B_{ik} = 1 \quad \forall i$$ (where $M$ is the total number of unique observations)

### 3. Initial state distribution matrix ($π$)
- Defines the probability distribution of the starting hidden state at the first time step $t=1$.
- $\pi_i = P(S_1 = s_i)$ : The probability that the initial state is $s_i$.
- Since the model must start in one of the states, the sum of all initial probabilities must be equal to 1:
  $$\sum_{i=1}^{N} \pi_i = 1$$


## HMM solve 3 fundamental problems
1. What is the probability that the model generates a *specific sequence of observations* ?
    - Solved through `forward algorithm`
2. Given a sequence of observations, what is the most likely sequence of hidden states that generated them?
    - `Viterbi algorithm`
3. How to learn `transition` and `emission` probabilities from real observation data?
    - `Baum-Welch Algorithm(Expectation Maximization/EM)`


# Viterbi Algorithm
*What is the single most likely hidden path that lead here?*

The Viterbi algorithm finds the most likely sequence of hidden states for a sequence of observations in a Hidden Markov Model (HMM). This notebook implements the algorithm in log space to avoid numerical underflow, then runs a small weather example.

An HMM is specified by initial state probabilities, transition probabilities, and emission probabilities. Each probability array below follows the order in `states` and `observations`.

In [ ]:
import numpy as np

def viterbi(observations, states, start_probability, transition_probability, emission_probability):
    """Return the most likely hidden-state sequence and its log probability."""
    # Preflight tests
    observations = list(observations)
    states = list(states)
    if not observations:
        raise ValueError("observations must contain at least one item")
    if not states:
        raise ValueError("states must contain at least one state")

    n_states = len(states)
    n_steps = len(observations)
    start = np.asarray(start_probability, dtype=float)
    transition = np.asarray(transition_probability, dtype=float)
    emission = np.asarray(emission_probability, dtype=float)
    if start.shape != (n_states,):
        raise ValueError("start_probability must have one value per state")
    if transition.shape != (n_states, n_states):
        raise ValueError("transition_probability must be a square state-by-state matrix")
    if emission.shape[0] != n_states:
        raise ValueError("emission_probability must have one row per state")
    if np.any(start < 0) or np.any(transition < 0) or np.any(emission < 0):
        raise ValueError("probabilities cannot be negative")

    # Map each observation to its column in the emission matrix.
    observation_index = {observation: i for i, observation in enumerate(observation_symbols)}
    try:
        obs_indices = [observation_index[item] for item in observations]
    except (KeyError, TypeError) as exc:
        raise ValueError(f"unknown observation: {exc.args[0]}") from exc
    if emission.shape[1] != len(observation_symbols):
        raise ValueError("emission matrix columns must match observation_symbols")

    # log(0) is -inf, correctly making impossible paths have zero probability.
    with np.errstate(divide="ignore"):
        log_start = np.log(start)
        log_transition = np.log(transition)
        log_emission = np.log(emission)

    scores = np.full((n_steps, n_states), -np.inf)
    backpointers = np.zeros((n_steps, n_states), dtype=int)
    scores[0] = log_start + log_emission[:, obs_indices[0]]

    for t in range(1, n_steps):
        for current in range(n_states):
            candidates = scores[t - 1] + log_transition[:, current]
            backpointers[t, current] = np.argmax(candidates)
            scores[t, current] = (
                candidates[backpointers[t, current]]
                + log_emission[current, obs_indices[t]]
            )

    last_state = int(np.argmax(scores[-1]))
    best_log_probability = float(scores[-1, last_state])
    path_indices = [last_state]
    for t in range(n_steps - 1, 0, -1):
        path_indices.append(int(backpointers[t, path_indices[-1]]))
    path_indices.reverse()
    return [states[i] for i in path_indices], best_log_probability

## Code explanation
```py
  scores = np.full((n_steps, n_states), -np.inf)
  backpointers = np.zeros((n_steps, n_states), dtype=int)
  scores[0] = log_start + log_emission[:, obs_indices[0]]

  for t in range(1, n_steps):
      for current in range(n_states):
          candidates = scores[t - 1] + log_transition[:, current]
          backpointers[t, current] = np.argmax(candidates)
          scores[t, current] = (
              candidates[backpointers[t, current]]
              + log_emission[current, obs_indices[t]]
          )

  last_state = int(np.argmax(scores[-1]))
  best_log_probability = float(scores[-1, last_state])
  path_indices = [last_state]
  for t in range(n_steps - 1, 0, -1):
      path_indices.append(int(backpointers[t, path_indices[-1]]))
  path_indices.reverse()
  return [states[i] for i in path_indices], best_log_probability
```

Above section of code is the dynamic programming implementation of Viterbi Algorithm

- `scores` : number of observations(no. of steps($M$)) × number of states($N$)  
$$
\begin{array}{c|ccc}
 & S_1 & S_2 & .. & S_N \\
\hline
O_1 & 0.01 & 0.89 & .. & 0.44 \\
O_2 & 0.22 & 0.10 & .. & 0.55 \\
 : & : & : & : & : \\
O_M & 0.01 & 0.33 & .. & 0.66
\end{array}
$$

score matrix : states along rows, observations along columns
- Captures $P(St | O)$

$$
\begin{array}{cc}
  & \begin{array}{ccc} O_1 & O_2 & O_3 & O4 \end{array} \\
  \begin{array}{c} St_1 \\ St_2 \\ St_3 \end{array} &
  \begin{bmatrix}
    0.78 & 0.88 & 0.11 & 0.23\\
    0.11 & 0.10 & 0.14 & 0.12\\
    0.11 & 0.02 & 0.75 & 0.65
  \end{bmatrix}
\end{array}
$$

- `transition` : number of states × number of states
$$
\begin{array}{c|ccc}
 & S_1 & S_2 & .. & S_N \\
\hline
S_1 & 0.01 & 0.89 & .. & 0.44 \\
S_2 & 0.22 & 0.10 & .. & 0.55 \\
 : & : & : & : & : \\
S_N & 0.01 & 0.33 & .. & 0.66
\end{array}
$$
  - Maps **transition probabilities** between states
- `emission` : number of states × number of possible observations in observation superset
$$
\begin{array}{c|ccc}
 & O_1 & O_2 & .. & O_V \\
\hline
S_1 & 0.01 & 0.89 & .. & 0.44 \\
S_2 & 0.22 & 0.10 & .. & 0.55 \\
 : & : & : & : & : \\
S_N & 0.01 & 0.33 & .. & 0.66
\end{array}
$$
  - Maps **hidden states** to **observable symbols**
  - $P(O_k | S_i)$ : Probability of observation $O_k$ given state $S_i$


*NOTE*: Difference between $N$ and $M$
- $N$: Number of possible observations; Number of elements in the Set of all possible observations
  - Can be considered as the *Vocabulary size*. Set of possiblee observations are the vocabulary
  - Sequence of $M$ is sampled from set of $N$
- $M$: Number of observations in the sample
  - May not contain all possible observations
  - Many observations may repeat



### Steps
1. Initialize `scores[0]` ($P(S_i,O_0)$)
    ```python
    scores[0] = log_start + log_emission[:, obs_indices[0]]
    ```
    - `log_start` : Initial probability of states($π$)
    - `log_emission[:, obs_indices[0]]` : emission probability for $0^{th}$ observation
      - Very first column from the `emission` matrix
    - After the operation `scores[0]` contain $P(S_i,O_0)$
      - According to definition of `emission` matrix:
        - $P(O_k|S_i) = (i,k)^{th}$ element of `emission` matrix
      - $P(S_i) × P(O_0 | S_i) = P(S_i, O_0)$
    
    - Observe closely; We've the luxuary to do entire column operation, because we already knew `log_start`.
2. This is a combination of 2 steps:
    1. Calculate the best candidates for next `log_start` equivalent; Lets call it $S_{current}$.   
    2. Multiply them with emission probability
    - Instead of implementing them as two decoupled row operations, we combine them inside the same loop
    ```python
      candidates = scores[t - 1] + log_transition[:, current]
      backpointers[t, current] = np.argmax(candidates)
      scores[t, current] = (
          candidates[backpointers[t, current]]
          + log_emission[current, obs_indices[t]]
      )
    ```
    - First statement calculate the initial probability for the $t^{th}$ step for the state `current`. Second statement captures the highest probability candidate from the set of candidates. This candidate is chosen as the predecessor of `current` state.
    - $3^{rd}$ statement caculate the `score` by multiplying with `emission` probability

3. Backtracking

# Core viterbi recursive equation
$$
V_t(j) = \max_{i \in [1, 2, \dots, N]} \bigg( V_{t-1}(i) \cdot \underset{\text{transition matrix}}{\underline{P(S_j | S_i)}} \bigg) \cdot \underset{\text{emission matrix}}{\underline{P(O_t | S_j)}}
$$

- $t=0$ is a special case

$$V_0(j) = \max\limits_{i \in [1,2, \dots N]}\bigg( \underbrace{V_{0-1}(i)}_\text{undefined} \cdot P(S_j | S_i) \bigg) \cdot P(O_0 | S_t) = \pi_j \cdot P(O_0 | S_j)$$

## Key takeaways
- `transition` matrix manages structural flow
  - Evaluate all possible histories that lead to `current` state
  - Choose the most likely history; the `max` function does this
- `emission` matrix evaluates the evidence
  - Adjust incoming path score based on conditional probability
  - Given the current hidden state, what is the probability of the observation

- Score of any state at time step t is determined by the probability distribution of states in previous time step and transition probabilities.

### Why the last point is important?
Lets explore an alternative possibility:
Assume `current` state calculation algorithm depends only on the highest probability(scored) state from previous time step.
Then the equation becomes:
$$ V_t(j) = \bigg(V_{t-1}(i) \cdot P(S_j | S_i)\bigg) \cdot P(O_t|S_j)$$
where $i$ is the highest scored state at time step $t-1$.

Above equation depends only on the best scored state from previous time step for calculating the next state. All other joint distributions(between score and transition matrix) are ignored.

Instead, Viterbi values score of all possible joint distributions. Then finds the best candidate and mutliply it with the emission matrix to find the score for current state.

Highest score in a timestep does not imply that state is chosen for that time step.

Chosen states of any time step are captured in the backtrack pointer array.
Elements of backtrack pointer array are derived by multiplying the score of the state at step and the transition probability between the next state of interest and current state, then finding the maximum value:
$$\max_\limits{i \in [1,2, \dots N]}\bigg(V_{t-1}(i) \cdot P(S_j|S_i) \bigg)$$

Hence state at any time step t is determined only after the scores for next step is defined. Which is true for all the future steps until the last step.

Hence it is **impossible** to define the **state at any time step** until we reach the last step.

## Tug of war during Viterbi decoding

$$\log \text{score} = \underbrace{\max_{i} \Big( \log V_{t-1}(i) + \log P(S_j \mid S_i) \Big)}_{\text{What the transition model expects}} + \underbrace{\log P(O_t \mid S_j)}_{\text{What the observation actually says}}$$

- Noisy observations :
  - Emission model fails
  - Transition model wins
- Ambiguous transitions :
  - Emission model wins
  - Transition model fails
  - State from which multiple transitions are equally likely, given the history  

# Forward-Backward Algorithm


## 1. Forward algorithm

High level concept  : *What is the total probability of all hidden paths lead here?*

- Asks this question for all possible hidden states
- Let $Α$ be the state vector at time $t$
$$ Α = \begin{bmatrix}
α_1\\
α_2\\
\vdots\\
α_N
\end{bmatrix}
$$


Caculate joint probability of
1. Seeing partial observations upto time $t$ ($O_1, O_2, O_3 .... O_t$)
2. Ending up in state $s_j$

$$ α_t(j) = P(O_1, O_2, O_3 ... O_t, S_t=s_j) $$

Expand above equation using **total probability theorem**

$$ α_t(j) = \sum_{i}^{N} P(O_1, O_2, \dots,O_{t-1}, S_{t-1}=s_i, O_t, S_t=s_j)$$

*NOTE: $s_i$ is introduced as $S_{t-1}$ deliberately*

Now, lets split the equation using **chain rule of probability** and **Markov Property**

Step 1: Reorganize parameters of the equation chronologically using commutativity of joint probability distributions:
$$ α_t(j) = \sum_{i}^{N} P(O_1, O_2, \dots, O_{t-1}, S_{t-1},  O_t, S_t)$$

Step 2: Apply Markov property to split the distribution
- Past: joint distribution upto $t-1$
- The transition: $P(S_t=s_j | \text{everything in past})$
- The emission: $P(O_t | \text{everything in past and the new state})$
- Markov property
  - Transition independence: "Current state only depends on immediate previous state, not on the complete history"
    - $P(S_t=s_j | S_{t-1} = s_i)$
  - Emission independence: "Given current hidden state $S_t=s_j$, current observation depends only on the current hidden state. Not on past observations or past hidden state"
    - $P(O_t | S_t=s_j)$


$$ α_t(j) = \underbrace{\sum_{i}^{N} P(O_1, O_2, \dots, O_{t-1}, S_{t-1}=s_i)}_{\alpha_{t-1}(i)} \cdot \underbrace{P(s_j | s_i)}_\text{transition indepdnence} \cdot \underbrace{P(O_t | S_t = s_j)}_\text{emission independence}$$

Hence the final equation becomes:

$$ α_t(j) = \sum_{i}^{N} \underbrace{α_{t-1}(i)}_\text{past} \cdot \underbrace{P(s_j | s_i)}_\text{transition matrix} \cdot \underbrace{P(O_t | S_t = s_j)}_{\text{emission}}  $$

Just like the **Viterbi algorithm**, $α_0(j)$ is a special case. In fact, equation for $V_0(j)$ and $α_0(j)$ are same

$$α_0(j)= π_j \cdot P(O_0 | s_j) $$

Last step converts these system of equations into recursively computable


## Backward algorithm
Calculate the conditional probability of
1. Finding all observations $\{O_{t+1}, O_{t+2}, \dots, O_T\}$
2. Given $S_t = s_i$

$$ β_t(i) = P(O_{t+1},O_{t+2}, \dots ,O_T | S_t = S_j) $$

### The anchor point
- Every dynamic programming problem require an anchor point to start from
- In case of forward algorithm, anchor point is $t=0$
- For backward algorithm, anchor point is $t=T$

At T, there are no future steps left to observe.
Probability of observing nothing given current state is $s_i$ is 1

$$B_T(i) = 1 \text{(for all states i)}$$

This is the starting point. From here, backward algorithm recurse back, one step at a time from $t=T-1$ to $t=1$


### The recursive steps
Given a time step $t$ and state $S_t = s_j$, what is the probability of finding all future observations.

1. **The Transition:** You transition from your current state $s_i$ to some next state $s_j$: $P(S_{t+1} = s_j \mid S_t = s_i) \rightarrow A_{ij}$
2. **The Emission:** That next state $s_j$ emits the very next observation $O_{t+1}$: $P(O_{t+1} \mid S_{t+1} = s_j) \rightarrow B_{j, o_{t+1}}$
3. **The Rest of the Future:** From step $t+1$ onwards, you look at the future beta score already computed for that next state: $\beta_{t+1}(j)$

By applying the Law of Total Probability over all possible next states $s_j$, we sum them up:

$$\beta_t(i) = \sum_{j=1}^{N} \underbrace{P(S_j \mid S_i)}_{\text{Transition } A_{ij}} \cdot \underbrace{P(O_{t+1} \mid S_j)}_{\text{Emission } B_{j, O_{t+1}}} \cdot \underbrace{\beta_{t+1}(j)}_{\text{Future Beta}}$$

---

### 4. Comparing the Loop Mechanics: Forward vs. Backward

Notice how structurally similar the core math is to the Forward algorithm, but with the flow reversed:

| Feature | Forward Algorithm ($\alpha$) | Backward Algorithm ($\beta$) |
| --- | --- | --- |
| **Direction** | Forward ($1 \to T$) | Backward ($T-1 \to 1$) |
| **Anchor / Base Case** | Starts at $t=0$ using $π$ and emission of $O_1$ | Starts at $t=T$ with all values set to $1$ ($\log 0$) |
| **What flows through time?** | Accumulates the **past** | Accumulates the **future** |
| **Mathematical Combination** | $\alpha_{t-1}(i) \cdot A_{ij} \cdot B_{j, o_t}$ | $A_{ij} \cdot B_{j, o_{t+1}} \cdot \beta_{t+1}(j)$ |
| **Aggregation Operator** | `logsumexp` (Sum over past states) | `logsumexp` (Sum over future states) |

---

### 5. Why Do We Need the Backward Algorithm? (The Master Key)

Individually, the Forward algorithm tells you about the past, and Viterbi tells you the single best path. But neither of them can answer a crucial question:

> *"Looking at the **entire** dataset from start to finish, what is the probability that the system was in state $s_i$ at a specific intermediate time step $t$?"*

This is where the magic of combining **Forward ($\alpha$)** and **Backward ($\beta$)** happens.

If you multiply the Forward score and the Backward score for any state $s_i$ at time $t$:

$$\alpha_t(i) \cdot \beta_t(i) = \underbrace{P(O_1, \dots, O_t, S_t = s_i)}_{\text{Past \& Present (Forward)}} \cdot \underbrace{P(O_{t+1}, \dots, O_T \mid S_t = s_i)}_{\text{Future (Backward)}}$$

By the rules of probability, multiplying a joint probability of the past/present by the conditional probability of the future gives you the joint probability of the **entire sequence** conditioned on being in state $s_i$ at time $t$:

$$\alpha_t(i) \cdot \beta_t(i) = P(O_1, \dots, O_T, S_t = s_i)$$

If you divide this product by the total probability of all observations $P(O)$ (which you get easily from the Forward algorithm by summing the final $\alpha$ row), you get the **Posterior State Probability ($\gamma_t(i)$)**:

$$\gamma_t(i) = \frac{\alpha_t(i) \cdot \beta_t(i)}{\sum_{j=1}^{N} \alpha_T(j)}$$

This $\gamma_t(i)$ vector is the core engine behind **Baum-Welch (Expectation-Maximization / Problem 3)**. It tells the algorithm how to re-estimate and learn the transition ($A$) and emission ($B$) matrices from real-world data by revealing what hidden states *actually* most likely occurred across the timeline in hindsight.


## Footnotes
### Total probability theorem
Given:
- $\{{B_1, B_2, B_3,\dots , B_N}\}$ are mutually **Exclusive** and **Exhaustive**

$$P(A) = \sum_{i}^{N} P(A \text{ and } B_i)$$

### Chain rule of probability
- $P(A,B,C, …) = P(A) \cdot P(B|A) \cdot P(C | A,B) ⋅ …$

In [ ]:
# Scratchpad
data = np.random.random((2,3))
print(data)
print("first row:")
print(data[0])
print(data.shape)

[[0.28317427 0.59186993 0.29601556]
 [0.24783677 0.71372695 0.01028304]]
first row:
[0.28317427 0.59186993 0.29601556]
(2, 3)
